# Project: Smart Q&A Bot

A production-ready Q&A bot that combines all Section 1 concepts:
- `ChatPromptTemplate` for the system persona
- `with_structured_output()` for typed responses
- `@traceable` decorator for LangSmith observability
- Batch processing for parallel queries
- Graceful error handling

**Architecture:**
```
User question
     │
     ▼
ChatPromptTemplate  (system persona + user input)
     │
     ▼
ChatOpenAI.with_structured_output(QAResponse)
     │
     ▼
QAResponse(answer, confidence, reasoning, follow_ups, sources_needed)
```

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List
from langsmith import traceable, Client

## 1. Define the Response Schema

`QAResponse` is a Pydantic model that the LLM must fill. Every field has a description used to guide structured output.

In [ ]:
class QAResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question.")
    confidence: str = Field(description="Confidence level: high, medium, or low")
    reasoning: str = Field(description="The reasoning behind the answer provided.")
    follow_up_questions: List[str] = Field(
        description="A list of follow-up questions related to the topic.",
        default_factory=list,
    )
    sources_needed: bool = Field(
        description="Indicates whether sources are needed for the answer.",
        default=False,
    )

## 2. SmartQABot Class

Key design decisions:
- `with_structured_output(QAResponse)` — LLM always returns a typed object (not a string)
- `@traceable` — every call is logged in LangSmith with inputs, outputs, and latency
- `ask_batch` — uses `.batch()` for parallel requests, much faster than calling `.invoke()` in a loop
- `except` in `ask()` — returns a graceful `QAResponse` instead of crashing the caller

In [ ]:
class SmartQABot:
    def __init__(self, model_name: str = "gpt-4o-mini", temperature: float = 0.3):
        # Bind the Pydantic schema to the model
        self.model = ChatOpenAI(
            model=model_name,
            temperature=temperature,
        ).with_structured_output(QAResponse)

        self.prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                """You are a knowledgeable Q&A assistant.
                Your guidelines:
                - Answer questions accurately and concisely
                - Be honest about uncertainty - set confidence to 'low' if unsure
                - Provide clear reasoning for your answers
                - Suggest relevant follow-up questions
                - Indicate if external sources would help""",
            ),
            ("human", "{question}"),
        ])

        self.chain = self.prompt | self.model

    @traceable(name="ask_question", run_type="chain")
    def ask(self, question: str) -> QAResponse:
        try:
            return self.chain.invoke({"question": question})
        except Exception as e:
            # Graceful fallback — never crash the caller
            return QAResponse(
                answer="I'm sorry, I couldn't process your question at this time.",
                confidence="low",
                reasoning=str(e),
                follow_up_questions=["Could you please try again later?"],
                sources_needed=True,
            )

    @traceable(name="ask_batch", run_type="chain")
    def ask_batch(self, questions: List[str]) -> List[QAResponse]:
        """Ask multiple questions in parallel using .batch()."""
        inputs = [{"question": q} for q in questions]
        return self.chain.batch(inputs)

## 3. Demo — Single Questions

In [ ]:
bot = SmartQABot()

questions = [
    "What is the capital of France?",
    "Explain the theory of relativity.",
    "How does photosynthesis work?",
]

for question in questions:
    response = bot.ask(question)
    print(f"Q: {question}")
    print(f"   Answer:     {response.answer}")
    print(f"   Confidence: {response.confidence}")
    print(f"   Follow-ups: {response.follow_up_questions[:2]}")
    print()

## 4. Demo — Batch Processing (parallel)

In [ ]:
batch_questions = ["What is Python?", "What is JavaScript?", "What is Rust?"]

# .batch() sends all three requests concurrently
responses = bot.ask_batch(batch_questions)

for q, r in zip(batch_questions, responses):
    print(f"Q: {q}")
    print(f"   {r.answer[:100]}...")
    print(f"   Confidence: {r.confidence}")
    print()

## 5. Demo — Error Handling Edge Case

In [ ]:
# Even with a very long (edge-case) question, the bot handles gracefully
long_question = "What is " + "very " * 100 + "important?"

response = bot.ask(long_question)
print(f"Handled gracefully — confidence: {response.confidence}")
print(f"Answer: {response.answer[:100]}")

## 6. LangSmith Tracing

If `LANGSMITH_API_KEY` is set, every `@traceable` call is logged automatically. Flush ensures all traces are sent before the notebook exits.

Configure in `.env`:
```
LANGSMITH_API_KEY=your_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=Smart Q&A Bot
```

In [ ]:
if os.getenv("LANGSMITH_API_KEY"):
    Client().flush()
    print("Traces flushed to LangSmith")
else:
    print("No LangSmith key — skipping flush")